Discover Dataflows

In SDMX, a Dataflow is equivalent to:

"A dataset family"

Examples:

International Banking Statistics

Credit Statistics

Property Prices

Exchange Rates

In [1]:
import pandas as pd
import sdmx
from tabulate import tabulate 

In [11]:
# ----------------------------------------------------
# List all supported sources - the ones which are the identifiers of a supported SDMX source
# ----------------------------------------------------

print(sdmx.list_sources())

['ABS', 'ABS_JSON', 'AR1', 'BBK', 'BIS', 'COMP', 'ECB', 'EMPL', 'ESTAT', 'ESTAT3', 'ESTAT_COMEXT', 'GROW', 'ILO', 'IMF', 'IMF_DATA', 'IMF_DATA3', 'INEGI', 'INSEE', 'ISTAT', 'LSD', 'NB', 'NBB', 'OECD', 'OECD_JSON', 'SGR', 'SPC', 'STAT_EE', 'StatCan', 'UNESCO', 'UNICEF', 'UNSD', 'UY110', 'WB', 'WB_WDI']


In [ ]:
# ----------------------------------------------------
# Inspecting a source and its attributes
# ----------------------------------------------------

source = sdmx.source.get_source("BIS")
print(source)
print("=="*70)
print(dir(source))

Source(id='BIS', url='https://stats.bis.org/api/v1', name='Bank for International Settlements', headers={}, data_content_type=<DataContentType.XML: 3>, versions={<Version.2.1: 2.1>}, supports={<Resource.actualconstraint: 'actualconstraint'>: True, <Resource.agencyscheme: 'agencyscheme'>: False, <Resource.allowedconstraint: 'allowedconstraint'>: False, <Resource.attachementconstraint: 'attachementconstraint'>: False, <Resource.availableconstraint: 'availableconstraint'>: False, <Resource.categorisation: 'categorisation'>: True, <Resource.categoryscheme: 'categoryscheme'>: True, <Resource.codelist: 'codelist'>: True, <Resource.conceptscheme: 'conceptscheme'>: True, <Resource.contentconstraint: 'contentconstraint'>: True, <Resource.customtypescheme: 'customtypescheme'>: False, <Resource.data: 'data'>: True, <Resource.dataconsumerscheme: 'dataconsumerscheme'>: False, <Resource.dataflow: 'dataflow'>: True, <Resource.dataproviderscheme: 'dataproviderscheme'>: False, <Resource.datastructure: 

In [ ]:
# ----------------------------------------------------
# Connect to BIS SDMX API
# ----------------------------------------------------
client = sdmx.Client("BIS") 

In [3]:
# ----------------------------------------------------
# Retrieve all dataflows
# ----------------------------------------------------
response = client.dataflow() # or response = client.get("dataflow") # depending on the version of the API
flows = response.dataflow
print(f"Total Dataflows Found : {len(flows)}\n")

Total Dataflows Found : 29



In [4]:
print(type(flows))

<class 'sdmx.dictlike.DictLike'>


In [5]:
flows[1]

<DataflowDefinition BIS:WS_CBPOL(1.0): Central bank policy rates>

In [ ]:
# ----------------------------------------------------
# Convert into DataFrame
# ----------------------------------------------------

records = []

for flow_id, flow in flows.items():

    try:
        name = flow.name.en
    except Exception:
        name = str(flow.name)

    try:
        description = flow.description.en
    except Exception:
        description = ""

    try:
        structure = flow.structure.id
    except Exception:
        structure = ""

    records.append(
        {
            "Flow ID": flow_id,
            "Name": name,
            "Description": description,
            "Structure": structure,
        }
    )

df = pd.DataFrame(records)
df = df.sort_values("Flow ID").reset_index(drop=True)
df.head()

,Flow ID,Name,Description,Structure
0,BIS_REL_CAL,BIS_RELEASE_CALENDAR,,BIS_RELEASE_CALENDAR
1,WS_CBPOL,Central bank policy rates,The interest rate which best captures the mone...,BIS_CBPOL
2,WS_CBS_PUB,Consolidated banking,,BIS_CBS
3,WS_CBTA,Central bank total assets,Tracks the evolution of the size of central ba...,CBTA
4,WS_CPMI_CASHLESS,CPMI cashless payments,Payment statistics covering indicators of reta...,CPMI_CASHLESS


In [7]:
# ----------------------------------------------------
# Print Summary
# ----------------------------------------------------

print(tabulate(df.head(30), headers="keys", tablefmt="github", showindex=True))

print("\n")
print("=" * 70)
print(f"Total Dataflows : {len(df)}")
print("=" * 70)

|    | Flow ID          | Name                                         | Description                                                                                                                                                                                                                                                                              | Structure            |
|----|------------------|----------------------------------------------|------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|----------------------|
|  0 | BIS_REL_CAL      | BIS_RELEASE_CALENDAR                         |                                                                                                                                                                          

In [8]:
# ----------------------------------------------------
# Save
# ----------------------------------------------------

output_file = "../output/bis_dataflows.csv"

df.to_csv(output_file, index=False)

print(f"\nSaved to {output_file}")


Saved to ../output/bis_dataflows.csv


In [9]:
#----------------------------------------------------
# Statistics
# ----------------------------------------------------

print("\nSummary Statistics\n")

print(df.info())

# print("\nMissing Descriptions:", df["Description"].isna().sum())
print("\nMissing Descriptions :", (df["Description"] == "").sum())
print("Missing Structures :", (df["Structure"] == "").sum())


Summary Statistics

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 29 entries, 0 to 28
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Flow ID      29 non-null     object
 1   Name         29 non-null     object
 2   Description  29 non-null     object
 3   Structure    29 non-null     object
dtypes: object(4)
memory usage: 1.0+ KB
None

Missing Descriptions : 14
Missing Structures : 0


In [10]:
#----------------------------------------------------
# Keyword Search Utility
# ----------------------------------------------------


print("\nSearch BIS Dataflows")
print("-" * 40)

keyword = input("Enter keyword (or press Enter to skip): ").strip().lower()

if keyword:
    result = df[
        df["Flow ID"].str.lower().str.contains(keyword, na=False) |
        df["Name"].str.lower().str.contains(keyword, na=False) |
        df["Description"].str.lower().str.contains(keyword, na=False) | 
        df["Structure"].str.lower().str.contains(keyword, na=False)
    ]

    if len(result):
        print(tabulate(result, headers="keys", tablefmt="github", showindex=False))
    else:
        print("No matching dataflows found.")


Search BIS Dataflows
----------------------------------------
| Flow ID     | Name                 | Description   | Structure            |
|-------------|----------------------|---------------|----------------------|
| BIS_REL_CAL | BIS_RELEASE_CALENDAR |               | BIS_RELEASE_CALENDAR |


In [ ]:
# Answers to some queries
Q1: client = sdmx.Client("BIS") # Question - What if the client name is something different than BIS ? 
Q2: How dow we know how to convert the data to a dataframe ?

Answers to some queries
Q1: What is a Dataflow?

A Dataflow is an SDMX object that represents a published statistical dataset. It defines what dataset is available (e.g., Credit Statistics, Exchange Rates) and links it to its underlying Data Structure Definition (DSD), which describes how the data is organized.

Reference:

[SDMX – What is SDMX?] Link : https://sdmx.org/about-sdmx/welcome/?utm_source=chatgpt.com
[BIS – SDMX Overview] Link : https://www.bis.org/statistics/sdmx.htm?utm_source=chatgpt.com




Q2: client = sdmx.Client("BIS") — What if the client name is something different than "BIS"?

"BIS" is the identifier of a supported SDMX data source. You can replace it with another supported source such as "ECB", "IMF", "OECD", "WB" (World Bank), etc., provided the sdmx library knows about that source. You can list all supported sources using:

import sdmx

print(sdmx.list_sources())

The client name tells the library which SDMX API endpoint to connect to.
Reference:

[sdmx1 Documentation – Data Sources] Link :  https://sdmx1.readthedocs.io/en/latest/sources.html?utm_source=chatgpt.com

Q3: How do we know how to convert the data to a DataFrame?

The SDMX API returns Python objects, not DataFrames. You inspect these objects (using methods like type(), dir(), or by reading the library documentation) and extract the fields you need into a list of dictionaries, which can then be passed to pandas.DataFrame(). This is a standard Python approach, not something specific to SDMX. The sdmx library also provides utilities to convert certain SDMX objects directly to pandas objects where appropriate.

Reference:

[sdmx1 Project Documentation] Link : https://pypi.org/project/sdmx1/?utm_source=chatgpt.com

Q1. Can we get more information on the sources printed by print(sdmx.list_sources())?

Yes. Each source has associated metadata such as its SDMX API endpoint, supported SDMX version(s), and capabilities (e.g., whether it supports data, metadata, or structure queries). You can inspect an individual source like this:

import sdmx

source = sdmx.source.get_source("BIS")

print(source)

or explore its attributes:

print(dir(source))

This helps you understand what the source supports before making requests.

Reference:

https://sdmx1.readthedocs.io/en/latest/sources.html

Q2. What is sdmx vs sdmx1 in Python?

There is one actively maintained Python package whose package name on PyPI is sdmx1, but you import it in Python as:

import sdmx

So:

sdmx1 → the package name you install using pip
sdmx → the module name you import in your code

For example:

pip install sdmx1
import sdmx

This naming avoids conflicts with an older, unmaintained package that was published under the name sdmx.

Reference:

https://pypi.org/project/sdmx1/
https://sdmx1.readthedocs.io/en/latest/

Note: In future major releases, the maintainers may eventually publish the package under the name sdmx, but as of today, the recommended installation is still pip install sdmx1 while importing it as import sdmx.